In [1]:
import pandas as pd

In [16]:
def df_transform(excel_path, sheet_name):
    
    # Transform the data: pivot from week columns to stage gate columns
    # The new table will have: Unique identifier, stage gate 1, 2, 3, 4 columns with week numbers as values

    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    # Identify the week columns (numeric columns from 1 to 52)
    week_columns = [col for col in df.columns if isinstance(col, int)]

    # Create a new list to store transformed rows
    transformed_data = []

    # Iterate through each row
    for idx, row in df.iterrows():
        unique_id = row['Work Package']
        
        # Create a dictionary to store stage gate -> week number mapping
        stage_gate_dict = {'work_package_name': unique_id}
        
        # Find the week number for each stage gate (1, 2, 3, 4)
        for week in week_columns:
            stage_gate_value = row[week]
            # If the value is a stage gate number (1-4), record the week number
            if stage_gate_value in [1, 2, 3, 4]:
                stage_gate_key = f'planned_gate_{int(stage_gate_value)}'
                stage_gate_dict[stage_gate_key] = week
        
        transformed_data.append(stage_gate_dict)

    # Create new DataFrame from transformed data
    df_transformed = pd.DataFrame(transformed_data)

    # Reorder columns: Unique identifier first, then Stage Gate 1, 2, 3, 4
    column_order = ['work_package_name', 'planned_gate_1', 'planned_gate_2', 'planned_gate_3', 'planned_gate_4']
    df_transformed = df_transformed[[column for column in column_order if column in df_transformed.columns]]
    
    # append the actual gate columns to the transformed dataframe
    for gate_num in range(1, 5):
        actual_gate_col = f'actual_gate_{gate_num}'
        planned_gate_col = f'planned_gate_{gate_num}'
        
        if planned_gate_col in df_transformed.columns:
            df_transformed[actual_gate_col] = df_transformed[planned_gate_col].apply(lambda x: row[x] if pd.notna(x) and x in week_columns else None)
    return df_transformed

In [17]:
df_path='../data/MASTER 2026 Delivery Programme.xlsx'
sheet_name='Sheet1'
df_transformed = df_transform(df_path, sheet_name)
df_transformed.head(5)

,work_package_name,planned_gate_1,planned_gate_2,planned_gate_3,planned_gate_4,actual_gate_1,actual_gate_2,actual_gate_3,actual_gate_4
0,N Yorks - 01,11.0,27.0,28.0,44.0,NaN,NaN,NaN,NaN
1,N Yorks - 02,11.0,33.0,34.0,NaN,NaN,NaN,NaN,NaN
2,N Yorks - 03,11.0,39.0,40.0,NaN,NaN,NaN,NaN,NaN
3,Blackburn & Darwen - 01,NaN,9.0,15.0,30.0,NaN,NaN,NaN,NaN
4,Blackburn & Darwen - 02,1.0,14.0,20.0,33.0,NaN,NaN,NaN,NaN


In [ ]:
# df_transformed.to_excel('../data/stage_gate_transformed.xlsx', index=False)